# Image Classification Tutorial
This notebook walks through a PyTorch image classification baseline step by step.

## 1. Setup and configuration
Set seeds, hyperparameters

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
import lightning as L

SEED = 42
BATCH_SIZE = 256
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
IMG_SIZE = 224
NUM_WORKERS = 2

VAL_RATIO = 0.1
TEST_RATIO = 0.1
TRAIN_RATIO = 1.0 - VAL_RATIO - TEST_RATIO

BACKBONE_WEIGHTS = models.ResNet50_Weights.IMAGENET1K_V2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "train"
print(f"Using CUDA: {torch.cuda.is_available()}")
print(f"Data directory: {DATA_DIR}")

Using CUDA: True
Data directory: ../data


## 2. Data loading and split
Load the images and encapsulate train, validation, and test splits in a LightningDataModule.

In [3]:
def build_eval_transform(weights, image_size):
    return weights.transforms(crop_size=image_size, resize_size=image_size)


def build_train_transform(weights, image_size):
    eval_t = build_eval_transform(weights, image_size)
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=eval_t.mean, std=eval_t.std),
        ]
    )


train_transform = build_train_transform(BACKBONE_WEIGHTS, IMG_SIZE)
eval_transform = build_eval_transform(BACKBONE_WEIGHTS, IMG_SIZE)

base_train_dataset = datasets.ImageFolder(TRAIN_DIR)
num_classes = len(base_train_dataset.classes)
idx_to_class = {idx: cls_name for cls_name, idx in base_train_dataset.class_to_idx.items()}


class TransformSubset(Dataset):
    def __init__(self, image_folder_dataset, indices, transform):
        self.dataset = image_folder_dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        real_index = self.indices[item]
        path, target = self.dataset.samples[real_index]
        image = self.dataset.loader(path)
        if self.transform is not None:
            image = self.transform(image)
        return image, target

In [4]:
class ImageDataModule(L.LightningDataModule):
    def __init__(
        self,
        base_dataset,
        train_transform,
        eval_transform,
        batch_size,
        num_workers,
        train_ratio,
        val_ratio,
        seed,
    ):
        super().__init__()
        self.base_dataset = base_dataset
        self.train_transform = train_transform
        self.eval_transform = eval_transform
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.seed = seed

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def setup(self, stage=None):
        n_samples = len(self.base_dataset)
        train_size = int(n_samples * self.train_ratio)
        val_size = int(n_samples * self.val_ratio)
        test_size = n_samples - train_size - val_size

        generator = torch.Generator().manual_seed(self.seed)
        train_subset, val_subset, test_subset = random_split(
            self.base_dataset,
            [train_size, val_size, test_size],
            generator=generator,
        )

        self.train_dataset = TransformSubset(self.base_dataset, train_subset.indices, self.train_transform)
        self.val_dataset = TransformSubset(self.base_dataset, val_subset.indices, self.eval_transform)
        self.test_dataset = TransformSubset(self.base_dataset, test_subset.indices, self.eval_transform)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )


datamodule = ImageDataModule(
    base_dataset=base_train_dataset,
    train_transform=train_transform,
    eval_transform=eval_transform,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
)
datamodule.setup()

In [5]:
print(f"Classes: {num_classes}")
print(f"Train samples: {len(datamodule.train_dataset)}")
print(f"Val samples: {len(datamodule.val_dataset)}")
print(f"Test samples (from train split): {len(datamodule.test_dataset)}")

Classes: 85
Train samples: 2992
Val samples: 374
Test samples (from train split): 374


## 3. Model definition
Define a custom `nn.Module` around a pretrained ResNet50 backbone.

In [6]:
class ClassificationModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        backbone = models.resnet50(weights=BACKBONE_WEIGHTS)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

In [7]:
class LitClassifier(L.LightningModule):
    def __init__(self, num_classes: int, learning_rate: float):
        super().__init__()
        self.save_hyperparameters()
        self.model = ClassificationModel(num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = (preds == labels).float().mean()

        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=1)
        acc = (preds == labels).float().mean()

        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_acc", acc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


lightning_model = LitClassifier(num_classes=num_classes, learning_rate=LEARNING_RATE)

## 4. Training loop
Train with PyTorch Lightning `Trainer.fit` using the `ImageDataModule`.

In [8]:
trainer = L.Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator="auto",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    deterministic=True,
    enable_model_summary=False,
)

trainer.fit(lightning_model, datamodule=datamodule)

metrics = trainer.callback_metrics
history_df = pd.DataFrame(
    [
        {
            "train_loss": float(metrics.get("train_loss", torch.tensor(float("nan"))).cpu()),
            "train_acc": float(metrics.get("train_acc", torch.tensor(float("nan"))).cpu()),
            "val_loss": float(metrics.get("val_loss", torch.tensor(float("nan"))).cpu()),
            "val_acc": float(metrics.get("val_acc", torch.tensor(float("nan"))).cpu()),
        }
    ]
)
history_df

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/workspace/modelretrieval-1-subtask-b/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 4: 100%|██████████| 12/12 [00:27<00:00,  0.43it/s, val_loss=1.110, val_acc=0.746, train_loss=1.510, train_acc=0.843] 

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 12/12 [00:27<00:00,  0.43it/s, val_loss=1.110, val_acc=0.746, train_loss=1.510, train_acc=0.843]


,train_loss,train_acc,val_loss,val_acc
0,1.511427,0.84258,1.111674,0.745989


## 5. Metric utilities
Reusable MRR helper functions

In [9]:
def ranked_predictions_from_logits(logits):
    return logits.argsort(dim=1, descending=True).cpu().tolist()


def mrr(y_true, ranked_predictions, return_details=False):
    reciprocal_ranks = []

    for true_item, predictions in zip(y_true, ranked_predictions):
        rank = next((index for index, item in enumerate(predictions) if item == true_item), None)
        reciprocal_ranks.append(1.0 / (rank + 1) if rank is not None else 0.0)

    score = float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0
    if return_details:
        return score, reciprocal_ranks
    return score

## 6. Final evaluation with MRR
Run the  evaluation helper on the held-out split and display the score.

In [10]:
@torch.no_grad()
def evaluate_mrr(model, loader, idx_to_class):
    model.eval()
    device = model.device
    true_labels = []
    ranked_predictions = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        ranked_predictions.extend(ranked_predictions_from_logits(logits))
        true_labels.extend(labels.cpu().tolist())

    mrr_score, reciprocal_ranks = mrr(true_labels, ranked_predictions, return_details=True)
    top1_predictions = [predictions[0] for predictions in ranked_predictions]
    split_result_df = pd.DataFrame(
        {
            "true_label": [idx_to_class[int(i)] for i in true_labels],
            "pred_label": [idx_to_class[int(i)] for i in top1_predictions],
            "reciprocal_rank": reciprocal_ranks,
        }
    )
    return mrr_score, split_result_df

In [11]:
mrr_score, split_result_df = evaluate_mrr(
    lightning_model,
    datamodule.test_dataloader(),
    idx_to_class,
)
print(f"Mean Reciprocal Rank (MRR): {mrr_score:.4f}")
split_result_df.head()

Mean Reciprocal Rank (MRR): 0.8364


,true_label,pred_label,reciprocal_rank
0,69,69,1.0
1,67,67,1.0
2,34,34,1.0
3,76,76,1.0
4,64,64,1.0


## 7. Read the results
Inspect the final MRR score and the per-sample reciprocal ranks.

In [12]:
print(f"Held-out split MRR: {mrr_score:.4f}")
print(f"Mean reciprocal rank per sample: {split_result_df['reciprocal_rank'].mean():.4f}")

Held-out split MRR: 0.8364
Mean reciprocal rank per sample: 0.8364
